# 🚀 ConfereAI - Fast Training (GPU Edition)
Este notebook permite treinar o motor neural do ConfereAI utilizando a GPU gratuita do Google Colab. 

**Instruções:**
1. Vá em `Ambiente de Execução` > `Alterar tipo de ambiente` e selecione **T4 GPU**.
2. Preencha as configurações abaixo.
3. Execute as células em ordem.

In [ ]:
# @title 1. Instalar Dependências
!pip install -q transformers[torch] librosa soundfile huggingface_hub accelerate

In [ ]:
# @title 2. Configurações do Hugging Face
HF_TOKEN = "" # @param {type:"string"}
REPO_ID = "TEDDyx86/confereai-dev" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}

from huggingface_hub import HfApi, login
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("❌ Por favor, insira o seu HF_TOKEN!")

In [ ]:
# @title 3. Upload do Dataset (.zip)
from google.colab import files
import zipfile
import os
import shutil

uploaded = files.upload()
dataset_zip = list(uploaded.keys())[0]

DATASET_DIR = "dataset_training"
if os.path.exists(DATASET_DIR): shutil.rmtree(DATASET_DIR)
os.makedirs(DATASET_DIR)

with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
    zip_ref.extractall(DATASET_DIR)

print(f"✅ Dataset extraído em: {DATASET_DIR}")

In [ ]:
# @title 4. Executar Treinamento (Fine-Tuning)
import torch
from torch.utils.data import Dataset
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2ForSequenceClassification, Trainer, TrainingArguments
import librosa

BASE_MODEL = "HyperMoon/wav2vec2-base-960h-finetuned-deepfake"
OUTPUT_DIR = "local_finetuned_model"

class DeepfakeDataset(Dataset):
    def __init__(self, root_dir, processor):
        self.files = []
        self.processor = processor
        for label, folder in enumerate(['real', 'fake']):
            path = os.path.join(root_dir, folder)
            if os.path.exists(path):
                for f in os.listdir(path):
                    if f.endswith(('.wav', '.mp3', '.flac')):
                        self.files.append({"path": os.path.join(path, f), "label": label})

    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        item = self.files[idx]
        speech, _ = librosa.load(item["path"], sr=16000)
        input_values = self.processor(speech, sampling_rate=16000, return_tensors="pt", padding="max_length", max_length=160000, truncation=True).input_values[0]
        return {"input_values": input_values, "labels": torch.tensor(item["label"], dtype=torch.long)}

processor = Wav2Vec2FeatureExtractor.from_pretrained(BASE_MODEL)
model = Wav2Vec2ForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2, ignore_mismatched_sizes=True)

# Congelar base para focar no aprendizado das novas fraudes (Lógica Robusta)
if hasattr(model, 'freeze_feature_extractor'):
    model.freeze_feature_extractor()
elif hasattr(model, 'freeze_feature_encoder'):
    model.freeze_feature_encoder()

if hasattr(model, 'wav2vec2'):
    for param in model.wav2vec2.parameters(): param.requires_grad = False

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    logging_steps=1,
    push_to_hub=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=DeepfakeDataset(DATASET_DIR, processor)
)

print("🚀 Iniciando treinamento na GPU...")
trainer.train()

model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"✅ Treinamento concluído. Modelo salvo em {OUTPUT_DIR}")

In [ ]:
# @title 5. Sincronizar com Hugging Face Space
api = HfApi()
print(f"📦 Subindo modelo para {REPO_ID}...")

api.upload_folder(
    folder_path=OUTPUT_DIR,
    path_in_repo=OUTPUT_DIR,
    repo_id=REPO_ID,
    repo_type="space",
    token=HF_TOKEN,
    commit_message="🤖 Auto-Update: Novo modelo treinado via Google Colab"
)

print("✨ Sucesso! O seu Space irá reiniciar em breve com o novo modelo.")